# Albedo und Indizes: Skalierung der Zusatzzweige

**Autor:** jd · **Datum:** 2026-10-12

**Ziel:** Prüfen, ob die neuen Zweige `use_mean_albedo` und `use_vegetation_indices` der hierarchischen RBF-SVM helfen – und ob ihre fehlende Skalierung schadet (#119).

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from awp2.config import BAND_PATTERN, MEAN_ALBEDO_COL
from awp2.data import load_folds, load_split
from awp2.evaluation import scorer
from awp2.models import make_hierarchical_svm
from awp2.preprocessing import PreprocessingConfig, build_preprocessor

In [ ]:
split = load_split()
split.X_train.shape

## Hypothese

Die Zweige für die mittlere Albedo und die Vegetationsindizes laufen in `build_preprocessor`
parallel zu den Spektren und werden **nicht** skaliert. Bei `use_spectral_standard_scale=True`
haben die Spektren je Band Streuung 1, die Albedo dagegen die Streuung der Reflektanz in %. Für
eine RBF-SVM bestimmt die Größenordnung, wie stark ein Merkmal in den Abstand eingeht – die
Albedo würde ihn dominieren, die Indizes gingen unter.

Modell: die getunte hierarchische RBF-SVM aus #87 (`hierarchical_svm`). Bewertet wird nur per CV
auf den Trainings-Folds, ohne MLflow und ohne Validierung. Die Variante „Zusatzzweige skaliert“
hängt im Notebook einen `StandardScaler` an diese Zweige – nur zur Demonstration; die eigentliche
Korrektur gehört in `build_preprocessor` (#119).

In [ ]:
model = make_hierarchical_svm().set_params(
    crop_estimator__C=30.0,
    crop_estimator__gamma=0.01,
    stage_estimator__C=100.0,
    stage_estimator__gamma=0.003,
)
EXTRA_BRANCHES = ("albedo", "indices")
base = {"use_spectral_standard_scale": True}
variants = {
    "Basis": {},
    "+ Albedo": {"use_mean_albedo": True},
    "+ Indizes": {"use_vegetation_indices": True},
    "+ Albedo + Indizes": {"use_mean_albedo": True, "use_vegetation_indices": True},
    "SNV + Albedo": {"use_spectral_snv": True, "use_mean_albedo": True},
    "SNV + Albedo + Indizes": {
        "use_spectral_snv": True,
        "use_mean_albedo": True,
        "use_vegetation_indices": True,
    },
}


def with_scaled_extras(preprocessor: ColumnTransformer) -> ColumnTransformer:
    preprocessor.transformers = [
        (name, Pipeline([*step.steps, ("scale", StandardScaler())]), columns)
        if name in EXTRA_BRANCHES
        else (name, step, columns)
        for name, step, columns in preprocessor.transformers
    ]
    return preprocessor.set_output(transform="pandas")


def cv_score(config: PreprocessingConfig, scale_extras: bool) -> pd.Series:
    preprocessor = build_preprocessor(config)
    if scale_extras:
        preprocessor = with_scaled_extras(preprocessor)
    pipeline = Pipeline([("preprocess", preprocessor), ("model", clone(model))])
    scores = cross_val_score(
        pipeline, split.X_train, split.y_train, cv=load_folds(), scoring=scorer()
    )
    return pd.Series({"CV": scores.mean(), "Std": scores.std()})

## Größenordnung der Zusatzmerkmale

In [ ]:
features = build_preprocessor(
    PreprocessingConfig(**base, use_mean_albedo=True, use_vegetation_indices=True)
).fit_transform(split.X_train)
spectral = features.filter(regex=BAND_PATTERN)
pd.DataFrame(
    {
        "Std": {
            "Spektralband (Median)": spectral.std().median(),
            MEAN_ALBEDO_COL: features[MEAN_ALBEDO_COL].std(),
            "NDVI": features["NDVI"].std(),
        }
    }
).round(3)

## CV je Variante

In [ ]:
results = pd.DataFrame(
    {
        (label, scaled): cv_score(PreprocessingConfig(**base, **extra), scaled)
        for label, extra in variants.items()
        for scaled in ((False,) if not extra else (False, True))
    }
).T
results.index = results.index.set_names(["Variante", "Zusatzzweige skaliert"])
results.round(4)

## Befund

- Unskaliert (Stand `main`) schadet die Albedo der SVM: 0.833 statt 0.839. Ihre Streuung (≈ 5)
  ist fünfmal so groß wie die eines standardisierten Bandes und dominiert den RBF-Abstand.
- Mit skalierten Zusatzzweigen ist der Schaden weg, ein Gewinn bleibt aus (≤ +0.001): Die Spektren
  sind je Band skaliert, die Helligkeit steckt also schon in ihnen.
- Nur mit SNV, das die Helligkeit entfernt, holt die Albedo einen Teil zurück (0.816 → 0.832),
  bleibt aber unter der Basis.
- Vorschlag (#119): Zusatzzweige mit derselben Skalierung wie die Spektren versehen.